# lint_tools

> MCP tools for linting notebooks for nbdev best practices

In [ ]:
#| default_exp tools.lint

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional

import re

from rich.panel import Panel
from mcp.server.fastmcp import FastMCP

from nbdev_mcp.utils.re import RELATIVE_IMPORT_PATTERN
from nbdev_mcp.utils.paths import (
    nbs_dir, settings_dict, resolve_selector, iter_notebooks,
    read_nb, write_nb, abs_module_for_nb, resolve_relative,
)
from nbdev_mcp.utils.nb import join_source, find_default_exp
from nbdev_mcp.utils.rich import render_table, render_panel

## Validate __init__ Notebooks

In [ ]:
#| export
def validate_inits(
    project: Optional[str] = None,
    fix: bool = False
) -> Dict[str, Any]:
    """Validate that every 00__init__ notebook has correct default_exp.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    fix : bool
        If True, automatically fix incorrect default_exp values.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'problems' list and 'fixed' count.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    nbs = nbs_dir(p)
    problems: List[Dict[str, Any]] = []
    fixed = 0
    
    for nb in iter_notebooks(p):
        if nb.name != '00__init__.ipynb':
            continue
        
        rel = nb.relative_to(nbs) if nbs in nb.parents else nb
        expected = '__init__' if len(rel.parts) == 1 else '.'.join(list(rel.parent.parts) + ['__init__'])
        
        data = read_nb(nb)
        found = find_default_exp(data)
        
        cell_idx, line_no = -1, -1
        for i, cell in enumerate(data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            for j, ln in enumerate(src.splitlines(), 1):
                if re.search(r'#\|\s*default_exp\s+', ln):
                    cell_idx, line_no = i, j
                    break
            if cell_idx != -1:
                break
        
        if found != expected:
            problems.append({
                'notebook': str(nb.relative_to(p)),
                'found': found,
                'expected': expected,
                'cell': cell_idx,
                'line': line_no
            })
            
            if fix:
                cells = data.get('cells', [])
                if cell_idx != -1:
                    src = join_source(cells[cell_idx].get('source', []))
                    new_src = re.sub(r'(#\|\s*default_exp\s+)[\w\.]+', f'\\1{expected}', src)
                    cells[cell_idx]['source'] = new_src.splitlines(True)
                else:
                    new_cell = {
                        'cell_type': 'code',
                        'metadata': {},
                        'source': [f'#| default_exp {expected}\n'],
                        'outputs': [],
                        'execution_count': None
                    }
                    cells.insert(0, new_cell)
                    data['cells'] = cells
                write_nb(nb, data)
                fixed += 1
    
    if problems:
        rows = [[pr['notebook'], str(pr['found']), pr['expected'], str(pr['cell']), str(pr['line'])]
                for pr in problems[:100]]
        pretty = render_table('__init__ default_exp issues', 
                              ['notebook', 'found', 'expected', 'cell', 'line'], rows)
    else:
        pretty = render_panel('validate_inits', 'All 00__init__ notebooks have correct default_exp')
    
    return {'ok': len(problems) == 0, 'problems': problems, 'fixed': fixed, 'pretty': pretty}

## Lint Rules

In [ ]:
#| export
def lint_rules(
    project: Optional[str] = None,
    fix_relative: bool = False
) -> Dict[str, Any]:
    """Lint notebooks for nbdev conventions.
    
    Checks for:
    - Manual __all__ definitions (not allowed)
    - Relative imports (should be absolute)
    - README.md being generated
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    fix_relative : bool
        If True, convert relative imports to absolute.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'issues' list and 'modified' count.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    s = settings_dict(p)
    lib = s.get('lib_name') or 'pkg'
    issues: List[Dict[str, Any]] = []
    changed = 0
    
    readme = p / 'README.md'
    if readme.exists():
        issues.append({
            'rule': 'readme_generated',
            'file': str(readme.relative_to(p)),
            'message': 'README.md is generated from nbs/index.ipynb — do not edit directly.'
        })
    
    for nb in iter_notebooks(p):
        data = read_nb(nb)
        mod = find_default_exp(data) or abs_module_for_nb(p, nb)[1]
        modified = False
        
        for i, cell in enumerate(data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            lines = src.splitlines()
            
            # Check for __all__ definitions
            if re.search(r'^\s*__all__\s*=', src, flags=re.MULTILINE):
                row_lines = [j for j, ln in enumerate(lines, 1) if re.match(r'\s*__all__\s*=', ln)]
                issues.append({
                    'rule': 'no_manual___all__',
                    'notebook': str(nb.relative_to(p)),
                    'cell': i,
                    'lines': row_lines,
                    'message': 'Never define __all__; nbdev auto-generates exports.'
                })
            
            # Check for relative imports
            new_lines: List[str] = []
            line_changed = False
            for ln in lines:
                m = RELATIVE_IMPORT_PATTERN.match(ln)
                if m:
                    target = m.group(1)
                    abs_mod = resolve_relative(mod, target)
                    abs_imp = f'from {lib}.{abs_mod} import {m.group(2)}'
                    issues.append({
                        'rule': 'no_relative_imports',
                        'notebook': str(nb.relative_to(p)),
                        'cell': i,
                        'line': len(new_lines) + 1,
                        'message': f"Relative import '{ln.strip()}' → '{abs_imp}'",
                        'suggestion': abs_imp
                    })
                    if fix_relative:
                        ln = abs_imp
                        line_changed = True
                new_lines.append(ln)
            
            if fix_relative and line_changed:
                cell['source'] = '\n'.join(new_lines).splitlines(True)
                modified = True
        
        if modified:
            write_nb(nb, data)
            changed += 1
    
    rows = []
    for it in issues[:200]:
        loc = it.get('file') or f"{it.get('notebook')}#cell{it.get('cell')}"
        rows.append([it.get('rule', ''), str(loc), it.get('message', '')])
    pretty = render_table('lint issues', ['rule', 'location', 'msg'], rows)
    
    if fix_relative:
        pretty += '\n' + render_panel('relative import fixes', f'Modified notebooks: {changed}')
    
    return {'ok': True, 'issues': issues, 'modified': changed, 'pretty': pretty}

In [ ]:
#| export
def lint_main_guards(project: Optional[str] = None) -> Dict[str, Any]:
    """Detect __main__ guards that would run during nbdev_prepare.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'issues' list of unsafe main guards.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    issues: List[Dict[str, Any]] = []
    
    for nb in iter_notebooks(p):
        data = read_nb(nb)
        for i, cell in enumerate(data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            lines = src.splitlines()
            
            has_guard = any(
                re.search(r'if\s+__name__\s*==\s*["\']__main__["\']', ln)
                for ln in lines
            )
            if not has_guard:
                continue
            
            has_eval_false = any('#| eval: false' in ln.lower() for ln in lines)
            has_export_main = any(
                re.match(r'\s*#\|\s*export\s+__main__', ln, flags=re.IGNORECASE)
                for ln in lines
            )
            
            if has_eval_false or has_export_main:
                continue
            
            issues.append({
                'notebook': str(nb.relative_to(p)),
                'cell': i,
                'has_eval_false': has_eval_false,
                'has_export_main': has_export_main,
                'message': "Add '#| eval: false' to prevent nbdev_prepare from running main()."
            })
    
    if issues:
        rows = [[it['notebook'], str(it['cell']), str(it['has_eval_false']), str(it['has_export_main'])]
                for it in issues[:200]]
        pretty = render_table('__main__ guard warnings', 
                              ['notebook', 'cell', 'eval_false', 'export_main'], rows)
    else:
        pretty = render_panel('lint_main_guards', 'No unsafe __main__ guards detected')
    
    return {'ok': len(issues) == 0, 'issues': issues, 'pretty': pretty}

## MCP Registration

In [ ]:
#| export
def add_lint_tools(mcp: FastMCP) -> None:
    """Attach linting tools to the MCP server.
    
    Parameters
    ----------
    mcp : FastMCP
        The MCP server instance.
    """
    mcp.add_tool(validate_inits)
    mcp.add_tool(lint_rules)
    mcp.add_tool(lint_main_guards)

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()